## CLI entry points

In [ ]:
#| default_exp cli

### Command wrappers

The implementation notebooks keep the notebook logic. This module owns the installed command-line surface: `@call_parse` entry points, script-shaped argument annotations, and command tracking.

In [ ]:
#| export
import json, os, re, signal, subprocess, time, traceback

from contextlib import contextmanager
from functools import wraps
from pathlib import Path

from fastcore.script import Param, call_parse

import nbskill.convert
import nbskill.execute
import nbskill.graph
import nbskill.knowledge
import nbskill.mcp
import nbskill.read
import nbskill.review
import nbskill.skill
import nbskill.workbench
import nbskill.write
from nbskill.foundation import failure_map_path, install_nbdev_pre_commit_hooks, load_failure_map


In [ ]:
#| export
_NBSKILL_HOOK_ROOTS = set()


In [ ]:
#| export
def _bump_count(data, kind, tool):
    counts = data.setdefault("counts", {})
    group = counts.setdefault(kind, {})
    group[tool] = group.get(tool, 0) + 1


In [ ]:
#| export
def _call_details(args, kwargs):
    details = {"cwd": str(Path.cwd())}
    if args and isinstance(args[0], (str, Path)): details["path"] = str(args[0])
    for key in ("path", "cell_id", "chapter"):
        value = kwargs.get(key)
        if value is not None: details[key] = str(value)
    return details


In [ ]:
#| export
def _write_failure_map(path, data):
    data["events"] = data.get("events", [])[-200:]
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=2, sort_keys=True), encoding="utf-8")


In [ ]:
#| export
def _record_tool_start(tool, details=None):
    path = failure_map_path()
    now = time.time()
    details = details or {}
    event = {"tool": tool, "ts": now, **details}
    try:
        data = load_failure_map(path)
        _bump_count(data, "usage", tool)
        last = data.get("last_call")
        if last:
            delta = now - float(last.get("ts", now))
            reasons = []
            if last.get("tool") == tool: reasons.append("same_tool")
            if delta <= 1.0: reasons.append("within_1s")
            if reasons:
                _bump_count(data, "friction", tool)
                data["events"].append({
                    "kind": "friction",
                    "tool": tool,
                    "path": details.get("path"),
                    "cell_id": details.get("cell_id"),
                    "previous_tool": last.get("tool"),
                    "previous_path": last.get("path"),
                    "seconds_since_previous": round(delta, 3),
                    "reasons": reasons,
                    "ts": now,
                })
        data["last_call"] = event
        _write_failure_map(path, data)
    except OSError:
        pass
    return event


In [ ]:
#| export
def _record_tool_failure(event, exc):
    path = failure_map_path()
    try:
        data = load_failure_map(path)
        tool = event["tool"]
        summary = "".join(traceback.format_exception_only(type(exc), exc)).strip()
        _bump_count(data, "failures", tool)
        data["events"].append({
            "kind": "failure",
            "tool": tool,
            "path": event.get("path"),
            "cell_id": event.get("cell_id"),
            "chapter": event.get("chapter"),
            "cwd": event.get("cwd"),
            "error_type": type(exc).__name__,
            "error": str(exc),
            "summary": summary,
            "ts": time.time(),
        })
        _write_failure_map(path, data)
    except OSError:
        pass


In [ ]:
#| export
@contextmanager
def _track_tool(tool, details=None):
    event = _record_tool_start(tool, details=details)
    try:
        yield
    except BaseException as exc:
        _record_tool_failure(event, exc)
        raise


In [ ]:
#| export
def _ensure_nbdev_pre_commit_hooks(path="."):
    if os.environ.get("NBSKILL_NO_INSTALL_HOOKS"): return None
    root = Path.cwd()
    if str(root) in _NBSKILL_HOOK_ROOTS: return None
    _NBSKILL_HOOK_ROOTS.add(str(root))
    try: return install_nbdev_pre_commit_hooks(path)
    except BaseException: return None


In [ ]:
#| export
def tracked_call(func):
    "Record one command-line tool call and ensure nbdev hooks are installed."
    @wraps(func)
    def wrapper(*args, **kwargs):
        _ensure_nbdev_pre_commit_hooks()
        with _track_tool(func.__name__, details=_call_details(args, kwargs)):
            return func(*args, **kwargs)
    return wrapper


In [ ]:
#| export
def _print_json(value):
    print(json.dumps(value, indent=2, sort_keys=True))
    return None


In [ ]:
#| export
def _print_workbench_result(result):
    text = result.get("rendered_plan") or result.get("summary") or json.dumps(result, indent=2, sort_keys=True)
    print(text)
    return None


In [ ]:
#| export
def _format_status(data):
    lines = [
        "nbskill status",
        f"version={data['version']}",
        f"cwd={data['cwd']}",
        f"python={data['python']}",
        f"mcp_command={data['mcp_command']}",
        f"mcp_command_path={data['mcp_command_path'] or '(not on PATH)'}",
        "cli_tools:",
    ]
    lines.extend(f"- {name}: {path or '(not on PATH)'}" for name, path in data["cli_tools"].items())
    lines.append(f"reconnect_hint={data['reconnect_hint']}")
    lines.append("install_commands:")
    lines.extend(f"- {cmd}" for cmd in data["install_commands"])
    return "\n".join(lines)


In [ ]:
#| export
@call_parse
@tracked_call
def context(
    target: str = "project",  # project, notebook path/name, chapter title, cell id, Python symbol, or literal search text
    scope: str = ".",  # Project, folder, glob, or notebook used to narrow target lookup
    overview: bool = False,  # True keeps compact overview; False shows fuller notebook markdown or related symbol context
):
    "Show the best notebook-aware context for one target."
    nbskill.read.context(target=target, scope=scope, overview=overview)

In [ ]:
#| export
@call_parse
@tracked_call
def write_nb(
    path: str,  # Notebook path, directory, or glob when replacing literals
    cells: Param("Cell block text", str, opt=False, nargs="?") = "",  # Cells to write; use - to read stdin
    cells_file: str | None = None,  # Read cell block text from a UTF-8 file to avoid shell escaping
    before_id: str | None = None,  # Insert before this stable cell id
    after_id: str | None = None,  # Insert after this stable cell id
    chapter: str | None = None,  # Chapter title string or regex; missing chapters are created
    replace: bool = False,  # Replace the full notebook, or the selected chapter body
    cell_type: str = "code",  # Default type for cells without %% marker
    run_test: bool = False,  # Execute the notebook with execnb after writing
    run_style: bool = False,  # Run chstyle after writing
    style_strict: bool = False,  # Fail when chstyle finds hints
    validate_code: bool = True,  # Validate new Python code cells before writing
    old_str: str | None = None,  # Literal text to replace across notebook cell sources
    new_str: str | None = None,  # Literal replacement text for old_str
    dry_run: bool = False,  # Show literal replacement plan without writing
    show_cells: bool = False,  # Include touched cell ids and compact diffs for literal replacements
):
    "Write cells to a notebook, or replace literal text across notebooks."
    return nbskill.write.write_nb(
        path, cells=cells, cells_file=cells_file, before_id=before_id, after_id=after_id, chapter=chapter,
        replace=replace, cell_type=cell_type, run_test=run_test, run_style=run_style, style_strict=style_strict,
        validate_code=validate_code, old_str=old_str, new_str=new_str, dry_run=dry_run, show_cells=show_cells,
    )

In [ ]:
#| export
@call_parse
@tracked_call
def update_cell(
    path: str,  # Notebook path
    new: Param("Replacement cell source, replacement text, or line-range replacement", str, opt=False, nargs="?") = "",
    new_file: str | None = None,  # Read replacement text from a UTF-8 file
    decode_newlines: bool = True,  # Decode literal \n sequences from CLI text
    cell_id: str | None = None,  # Stable cell id
    old_str: str | None = None,  # Literal text to replace in the selected cell
    line_range: str | None = None,  # 1-based line or range, such as 2 or 2:4
    split: bool = False,  # Split a multi-cell replacement into separate cells
    split_before: str | None = None,  # Split the existing cell before the first matching line
    cell_type: str = "code",  # Replacement cell type when parsing text
    run_test: bool = False,  # Execute the notebook after writing
    validate_code: bool = True,  # Validate replacement Python
    dry_run: bool = False,  # Show the edit without writing
):
    "Update one notebook cell by id, replace text/ranges, or split one cell."
    return nbskill.write.update_cell(
        path, new=new, new_file=new_file, decode_newlines=decode_newlines, cell_id=cell_id, old_str=old_str,
        line_range=line_range, split=split, split_before=split_before, cell_type=cell_type, run_test=run_test,
        validate_code=validate_code, dry_run=dry_run,
    )

In [ ]:
#| export
@call_parse
@tracked_call
def batch_edit_nb(
    plan: Param("JSON edit plan, or - to read stdin", str, opt=False, nargs="?") = "",
    plan_file: str | None = None,  # Read JSON edit plan from a UTF-8 file
    path: str | None = None,  # Default notebook path for operations without a path
    dry_run: bool = True,  # Show the edit plan without writing
    validate_code: bool = True,  # Validate replacement Python
    default_cell_type: str = "code",  # Default type for structured cells
):
    "Apply a JSON batch edit plan to one or more notebooks with locks, diffs, and read-back verification."
    return nbskill.write.batch_edit_nb(
        plan=plan, plan_file=plan_file, path=path, dry_run=dry_run, validate_code=validate_code,
        default_cell_type=default_cell_type,
    )

In [ ]:
#| export
@call_parse
@tracked_call
def split_nb_chapter(
    path: str,  # Source notebook path
    chapter: str,  # Chapter title string or regex to split out
    dest: str,  # Destination notebook path
    default_exp: str | None = None,  # Destination nbdev default_exp; defaults from dest path
    dry_run: bool = True,  # Show the split plan without writing notebooks
    force: bool = False,  # Overwrite dest if it already exists
    promote_private: bool = True,  # Promote referenced private source helpers by dropping the leading underscore
):
    "Split one ## chapter into a new nbdev notebook."
    return nbskill.write.split_nb_chapter(
        path, chapter, dest, default_exp=default_exp, dry_run=dry_run, force=force, promote_private=promote_private,
    )

In [ ]:
#| export
@call_parse
@tracked_call
def exec_nb(
    path: str,  # Notebook path
    up2id: int | str | None = None,  # Stop after this cell index or id
    chapter: str | None = None,  # Run one chapter by heading
    timeout: int = 30,  # Per-cell timeout in seconds
    show_output: bool = True,  # Print visible outputs after execution
    allow_new: bool = False,  # Permit new code cells without prior execution approval
    check_only: bool = False,  # Execute without writing outputs back
):
    "Execute a notebook with the normal safe defaults."
    return nbskill.execute.exec_nb(
        path, up2id=up2id, chapter=chapter, timeout=timeout, show_output=show_output,
        allow_new=allow_new, check_only=check_only,
    )

In [ ]:
#| export
@call_parse
@tracked_call
def diff_nb(
    path: str,  # Notebook path
    ref_a: str | None = "HEAD",  # Git ref for the left side; pass None to diff the file against itself
    ref_b: str | None = None,  # Git ref for the right side; defaults to working tree
    adds: bool = True,  # Include added cells
    changes: bool = True,  # Include changed cells
    dels: bool = False,  # Include deleted cells
    cell_id: str | None = None,  # Restrict output to one cell id
    after_id: str | None = None,  # Restrict output to cells after this id
):
    "Print nbdev-style diffs for code cells only; summarize nbskill metadata-only changes."
    return nbskill.review.diff_nb(
        path, ref_a=ref_a, ref_b=ref_b, adds=adds, changes=changes, dels=dels, cell_id=cell_id, after_id=after_id,
    )

In [ ]:
#| export
@call_parse
@tracked_call
def style_check(
    path: Param("File or folder to check", str, opt=False, nargs="?") = ".",  # File or folder to check
    skip_folder_re: str | None = None,  # Regex for folders to skip
    skip_path: str | None = None,  # Comma-separated paths to skip
    strict: bool = False,  # Exit non-zero when diagnostics are present
    delete_after_output: bool = False,  # Compatibility spelling for delete-after-output
    delete_after_outout: bool = False,  # Deprecated misspelling kept for CLI compatibility
    max_output_chars: int = 12000,  # Cap printed output
    max_diagnostics: int = 200,  # Cap structured diagnostics
    fix: bool = False,  # Apply safe automatic fixes
    dry_run: bool = True,  # Show fixes without applying them
    changed_only: bool = False,  # Restrict notebook style diagnostics to changed cells
    ref_a: str | None = "HEAD",  # Left git ref for changed-only mode
    ref_b: str | None = None,  # Right git ref for changed-only mode
):
    "Print capped fast.ai style hints, notebook hygiene warnings, and global tool usage."
    return nbskill.review.style_check(
        path=path, skip_folder_re=skip_folder_re, skip_path=skip_path, strict=strict,
        delete_after_output=delete_after_output, delete_after_outout=delete_after_outout,
        max_output_chars=max_output_chars, max_diagnostics=max_diagnostics, fix=fix, dry_run=dry_run,
        changed_only=changed_only, ref_a=ref_a, ref_b=ref_b,
    )

In [ ]:
#| export
@call_parse
@tracked_call
def nbskill_validate(
    path: Param("Notebook file, folder, or glob to validate", str, opt=False, nargs="?") = "nbs",
    strict: bool = True,  # Exit non-zero when validation problems are present
):
    "Validate nbskill metadata needed for safe notebook tools."
    return nbskill.review.validate_nbs(path=path, strict=strict)

In [ ]:
#| export
@call_parse
@tracked_call
def convert(
    path: str,  # Python file/folder, or existing Python project/package root
    mode: str = "notebook",  # notebook for file/folder conversion, project for nbdev project creation
    dest: str | None = None,  # Notebook path/output folder for notebook mode; project root for project mode
    nbs_path: str = "nbs",  # Destination notebooks folder
    recursive: bool = True,  # Search subfolders in notebook mode
    maxdepth: int | None = None,  # Maximum folder scan depth in notebook mode
    preserve_tree: bool = True,  # Preserve package folder structure under nbs_path
    class_lines: int = 100,  # Split classes larger than this line count
    method_lines: int = 10,  # Split methods larger than this out of large classes
    package: str | None = None,  # Package root name
    include: str | None = None,  # Comma-separated include globs
    exclude: str | None = None,  # Comma-separated exclude globs
    skip_init: bool = True,  # Skip __init__.py files in notebook mode
    include_tests: bool = False,  # Include test files in notebook mode
    dry_run: bool = False,  # Show planned writes without writing
    force: bool = True,  # Overwrite existing notebooks/files
    run_validation: bool = True,  # Run nbdev validation in project mode
):
    "Convert Python sources to nbdev notebooks or a small nbdev project."
    return nbskill.convert.convert(
        path, mode=mode, dest=dest, nbs_path=nbs_path, recursive=recursive, maxdepth=maxdepth,
        preserve_tree=preserve_tree, class_lines=class_lines, method_lines=method_lines,
        package=package, include=include, exclude=exclude, skip_init=skip_init, include_tests=include_tests,
        dry_run=dry_run, force=force, run_validation=run_validation,
    )

In [ ]:
#| export
@call_parse
@tracked_call
def build_nbskill_skill(readme_path: str = "README.md", out_path: str = "nbskill/SKILL.md"):
    "Build SKILL.md from the marked section of README.md."
    return nbskill.skill.build_skill_from_readme(readme_path=readme_path, out_path=out_path)

In [ ]:
#| export
@call_parse
@tracked_call
def install_nbskill(
    target: str = "codex",  # Target agent: codex or claude
    skills_dir: str | None = None,  # Explicit skills directory
    skill_name: str = "jupyter-notebooks",  # Installed skill folder name
    overwrite: bool = True,  # Overwrite an existing installation
    install_hooks: bool = False,  # Install nbdev pre-commit hooks
    restart_mcp: bool = True,  # Include restart guidance for running MCP servers
):
    "Install the bundled SKILL.md into a Codex or Claude Code skills directory."
    return nbskill.skill.install_nbskill(
        target=target, skills_dir=skills_dir, skill_name=skill_name, overwrite=overwrite,
        install_hooks=install_hooks, restart_mcp=restart_mcp,
    )

In [ ]:
#| export
@call_parse
@tracked_call
def symbol_connection(path: str = "nbs", start: str = "", end: str = "", max_depth: int = 6, json_output: bool = False):
    "Print the shortest static callee chain connecting two notebook symbols."
    return nbskill.graph.symbol_connection(path=path, start=start, end=end, max_depth=max_depth, json_output=json_output)

In [ ]:
#| export
@call_parse
@tracked_call
def reference(
    action: str = "query",  # add, list, ingest, or query
    query: str | None = None,  # Natural-language query for action=query
    top_k: int = 3,  # Number of implementation hits for action=query
    include_branch: bool = False,  # Include direct same-repo callers and callees
    current_repo: str = ".",  # Current project for dependency status
    repos: str | None = None,  # Optional repo name or comma-separated names to search
    url: str | None = None,  # Repository URL/path for action=add
    name: str | None = None,  # Reference name for add/ingest
    version: str = "HEAD",  # Git ref for action=add
    package: str | None = None,  # Package filter or package name
    path: str | None = None,  # Override reference home
    kind: str | None = None,  # Optional query filter: readme, module, function, class, method
    module: str | None = None,  # Optional module filter for action=query
    symbol: str | None = None,  # Optional symbol filter for action=query
    all: bool = False,  # Ingest all registered references
    force: bool = False,  # Re-ingest even when the indexed version matches
):
    "Manage and search reference implementations."
    action = str(action or "query").lower()
    if action == "add":
        if not url: raise ValueError("reference action='add' needs url")
        result = nbskill.knowledge.reference_add(url, name=name, version=version, package=package, path=path)
    elif action == "list":
        result = nbskill.knowledge.reference_list(path=path)
    elif action == "ingest":
        result = nbskill.knowledge.reference_ingest(name=name, all=all, path=path, force=force)
    elif action == "query":
        if not query: raise ValueError("reference action='query' needs query")
        result = nbskill.knowledge.reference_query(
            query, top_k=top_k, include_branch=include_branch, current_repo=current_repo, repos=repos, path=path,
            kind=kind, package=package, module=module, symbol=symbol,
        )
    else:
        raise ValueError("action must be add, list, ingest, or query")
    return _print_json(result)

In [ ]:
#| export
@call_parse
@tracked_call
def agent_workbench(
    goal: str,  # Desired software-development outcome
    notebook: str | None = None,  # Required when execute=True; also narrows context when provided
    contract_file: str | None = None,  # Optional JSON contract overrides
    execute: bool = False,  # Execute the rendered plan through execute_plan
    max_steps: int = 8,  # Maximum inner-agent steps when executing
    timeout: int = 30,  # Per-cell timeout for execute_plan
):
    "Prepare or execute a taste-aware, small-diff agent workbench run."
    result = nbskill.workbench.agent_workbench(
        goal, notebook=notebook, contract_file=contract_file, execute=execute,
        max_steps=max_steps, timeout=timeout,
    )
    return _print_workbench_result(result)


In [ ]:
#| export
@call_parse
@tracked_call
def nbskill_status(json_output: bool = False):  # Print JSON instead of text
    "Report nbskill version, MCP command setup, canonical CLI tools, and reconnect hints."
    data = nbskill.mcp.nbskill_status(json_output=json_output)
    print(json.dumps(data, indent=2, sort_keys=True) if json_output else _format_status(data))
    return None


In [ ]:
#| export
@call_parse
def nbskill_mcp(
    transport: str = "stdio",  # MCP transport; stdio is what Codex/Claude use for local servers
    show_banner: bool = False,  # Show FastMCP startup banner
):
    "Run the nbskill MCP server."
    return nbskill.mcp.main(transport=transport, show_banner=show_banner)

### MCP server process control
Codex owns stdio MCP server lifetimes, but nbskill can still make development safer by finding and stopping the per-project server processes it launched. Stopping the old process lets the MCP client reconnect to a fresh `nbskill_mcp` after exports or package updates, without changing the shared knowledge database.

In [ ]:
#| export
def _mcp_process_rows():
    proc = subprocess.run(["ps", "-axo", "pid=,ppid=,command="], text=True, capture_output=True)
    if proc.returncode:
        raise RuntimeError((proc.stderr or proc.stdout or "ps failed").strip())
    rows = []
    for line in proc.stdout.splitlines():
        parts = line.strip().split(None, 2)
        if len(parts) != 3: continue
        try: pid, ppid = int(parts[0]), int(parts[1])
        except ValueError: continue
        rows.append({"pid": pid, "ppid": ppid, "command": parts[2]})
    return rows
def _mcp_project_root(project=None):
    return Path(project or Path.cwd()).expanduser().resolve()
def _mcp_project_match_texts(project=None):
    raw = Path(project or Path.cwd()).expanduser()
    texts = {str(raw)}
    try: texts.add(str(raw.resolve()))
    except OSError: pass
    return texts
def _mcp_process_matches(row, project=None, all_projects=False):
    command = row.get("command", "")
    if row.get("pid") == os.getpid(): return False
    if not re.search(r"(^|[\s/])nbskill_mcp($|\s)", command): return False
    if all_projects: return True
    return any(text in command for text in _mcp_project_match_texts(project))
def _find_nbskill_mcp_processes(project=None, all_projects=False):
    return [row for row in _mcp_process_rows() if _mcp_process_matches(row, project=project, all_projects=all_projects)]
def _alive_pids(pids):
    current = {row["pid"] for row in _mcp_process_rows()}
    return [pid for pid in pids if pid in current]
def _stop_nbskill_mcp_processes(project=None, all_projects=False, timeout=5.0, force=True, dry_run=False):
    matches = _find_nbskill_mcp_processes(project=project, all_projects=all_projects)
    result = {
        "project": None if all_projects else str(_mcp_project_root(project)),
        "all_projects": bool(all_projects),
        "matched": matches,
        "terminated": [],
        "forced": [],
        "alive": [],
        "dry_run": bool(dry_run),
    }
    if dry_run or not matches: return result
    for row in matches:
        try:
            os.kill(row["pid"], signal.SIGTERM)
            result["terminated"].append(row["pid"])
        except ProcessLookupError:
            pass
    deadline = time.time() + float(timeout)
    pending = _alive_pids([row["pid"] for row in matches])
    while pending and time.time() < deadline:
        time.sleep(0.1)
        pending = _alive_pids(pending)
    if pending and force:
        for pid in pending:
            try:
                os.kill(pid, signal.SIGKILL)
                result["forced"].append(pid)
            except ProcessLookupError:
                pass
        pending = _alive_pids(pending)
    result["alive"] = pending
    return result
def _print_mcp_control_result(result, json_output=False):
    if json_output:
        print(json.dumps(result, indent=2, sort_keys=True))
        return None
    lines = [
        f"matched={len(result['matched'])}",
        f"terminated={len(result['terminated'])}",
        f"forced={len(result['forced'])}",
        f"alive={len(result['alive'])}",
    ]
    if result.get("project"): lines.insert(0, f"project={result['project']}")
    if result.get("dry_run"): lines.insert(0, "dry_run=true")
    for row in result["matched"]:
        lines.append(f"pid={row['pid']} ppid={row['ppid']} command={row['command']}")
    print("\n".join(lines))
    return None

In [ ]:
#| export
@call_parse
def nbskill_mcp_start(
    transport: str = "stdio",  # MCP transport; stdio is what Codex/Claude use for local per-project servers
    show_banner: bool = False,  # Show FastMCP startup banner
):
    "Run the nbskill MCP server in the foreground."
    return nbskill.mcp.main(transport=transport, show_banner=show_banner)
@call_parse
def nbskill_mcp_stop(
    project: str | None = None,  # Project root to target; defaults to the current working directory
    all_projects: bool = False,  # Stop nbskill MCP servers for every project
    timeout: float = 5.0,  # Seconds to wait after SIGTERM before forcing
    force: bool = True,  # Send SIGKILL to remaining matched processes after timeout
    dry_run: bool = False,  # Show matched processes without stopping them
    json_output: bool = False,  # Print JSON instead of text
):
    "Stop running nbskill MCP server processes for this project."
    result = _stop_nbskill_mcp_processes(project=project, all_projects=all_projects, timeout=timeout, force=force, dry_run=dry_run)
    return _print_mcp_control_result(result, json_output=json_output)
@call_parse
def nbskill_mcp_restart(
    project: str | None = None,  # Project root to target; defaults to the current working directory
    all_projects: bool = False,  # Restart nbskill MCP servers for every project
    timeout: float = 5.0,  # Seconds to wait after SIGTERM before forcing
    force: bool = True,  # Send SIGKILL to remaining matched processes after timeout
    dry_run: bool = False,  # Show matched processes without stopping them
    json_output: bool = False,  # Print JSON instead of text
):
    "Stop nbskill MCP processes so the MCP client starts a fresh per-project server."
    result = _stop_nbskill_mcp_processes(project=project, all_projects=all_projects, timeout=timeout, force=force, dry_run=dry_run)
    result["start"] = "stdio MCP servers are client-owned; Codex starts a fresh nbskill_mcp process on the next connection. Use nbskill_mcp_start to run one in the foreground."
    return _print_mcp_control_result(result, json_output=json_output)

In [ ]:
#| hide
sample = {"pid": 123, "ppid": 1, "command": "/opt/homebrew/bin/uv run --project /tmp/demo nbskill_mcp"}
assert _mcp_process_matches(sample, project="/tmp/demo")
assert not _mcp_process_matches(sample, project="/tmp/other")
assert not _mcp_process_matches({"pid": 124, "ppid": 1, "command": "nbskill_mcp_restart"}, all_projects=True)